# Dataset Preparation:

In this notebook we want to create a very complete and clean dataset in JSON format to use it for finetuning a smaller model.

## 1. Import dependencies:

Importing and installing the packages and libraries we need within this project.

In [1]:
!pip -q install requests jsonschema
!pip install -U langchain-openai langchain-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.8/84.8 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 490.2/490.2 kB 33.6 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.6
    Uninstalling langchain-core-1.2.6:
      Successfully uninstalled langchain-core-1.2.6


* `requests`: to send HTTP requests to the OpenRouter API and receive model responses

* `jsonschema`: to define a schema and validate that every JSON sample in our dataset has the correct structure

In [2]:
import os, json, re, textwrap, datetime, random
import requests
from jsonschema import Draft7Validator
import time
from collections import Counter
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage


* `os`: work with files and directories (saving/loading the dataset)

* `json`: read and write JSON files

* `re`: clean or transform text using regular expressions

* `textwrap`: format long strings (e.g., descriptions or code) for readability

* `datetime`: add timestamps or handle date-related fields in the dataset

* `Draft7Validator` (from jsonschema): validate each JSON object against our schema, so we can catch malformed samples early

## 2. API Key and Models:

In this step, we set up the API key to connect to OpenRouter and define a small set of LLM models that we’ll use in different stages of dataset generation.

In [3]:
OPENROUTER_API_KEY = "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"
MODELS = {
    "qwen": "qwen/qwen3-coder",
    "claude" : "anthropic/claude-sonnet-4.5",
    "gpt": "openai/gpt-oss-120b",
    "mistral": "mistralai/devstral-2512:free"
    }

## 3. Helper Functions

In this section we define the helper functions that will be reused across all parts of the dataset generation pipeline.


In [4]:
ITEM_SCHEMA = {
    "type": "object",
    "required": ["title", "description", "difficulty"],
    "properties": {
        "title": {
            "type": "string",
            "minLength": 3,
            "maxLength": 100,
        },
        "description": {
            "type": "string",
            "minLength": 20,
            "maxLength": 600,
        },
        "difficulty": {
            "type": "string",
            "enum": ["easy", "medium", "hard"],
        },
    },
    "additionalProperties": False,
}

In [5]:
def validate_items(arr, N=50):
    """
    Validate a list (or JSON string) of project objects.

    - Top-level must be an array.
    - Each item must match ITEM_SCHEMA.
    - Total length must be exactly N (default 50).
    - Titles must be unique (case-insensitive).

    Returns:
        list[str]: list of human-readable error messages (empty if valid).
    """

    errs = []

    # If arr is raw JSON text, parse it first
    if isinstance(arr, str):
        try:
            arr = json.loads(arr)
        except json.JSONDecodeError as e:
            return [f"Input is not valid JSON: {e}"]

    # Top-level type check
    if not isinstance(arr, list):
        return ["Top-level JSON value must be an array of objects."]

    # Schema validation (size + per-item schema)
    ARRAY_SCHEMA = {
        "type": "array",
        "items": ITEM_SCHEMA,
        "minItems": N,
        "maxItems": N,
    }

    errs.extend(e.message for e in Draft7Validator(ARRAY_SCHEMA).iter_errors(arr))

    # Explicit length check (gives a clearer message than JSON Schema sometimes)
    if len(arr) != N:
        errs.append(f"Array length must be exactly {N}, got {len(arr)}.")

    # Duplicate title check (robust to bad items)
    titles = []
    for i, x in enumerate(arr):
        if not isinstance(x, dict):
            errs.append(f"Item at index {i} is not an object.")
            continue
        title = (x.get("title") or "").strip().lower()
        titles.append(title)

    if len(titles) != len(set(titles)):
        errs.append("Duplicate titles detected.")

    return errs

In [6]:
def extract_json_array(text: str, return_candidate_on_fail: bool = False):
    """
    Try to extract and parse a JSON array from model output.

    Returns:
        - list (parsed JSON array) on success
        - None if nothing plausible found and return_candidate_on_fail=False
        - str (candidate JSON slice) if parsing fails and return_candidate_on_fail=True
    """
    # Strip ```json ... ``` fences if present
    m = re.search(r"```(?:json)?\s*([\s\S]*?)\s*```", text, re.IGNORECASE)
    if m:
        text = m.group(1).strip()

    start = text.find('[')
    if start == -1:
        return None

    depth = 0
    end = None
    for i in range(start, len(text)):
        ch = text[i]
        if ch == '[':
            depth += 1
        elif ch == ']':
            depth -= 1
            if depth == 0:
                end = i + 1
                break

    if end is None:
        return None

    candidate = text[start:end].strip()

    try:
        data = json.loads(candidate)
        if isinstance(data, list):
            return data
    except Exception as e:
        if return_candidate_on_fail:
            print(f"[extract_json_array] JSON parsing failed: {e}")
            return candidate
        return None


In [7]:
# Helper: convert OpenAI-style dict messages -> LangChain messages
def to_langchain_messages(messages):
    """
    Convert a list of {'role': 'system'|'user'|'assistant', 'content': '...'}
    into LangChain message objects.
    """
    lc_messages = []
    for m in messages:
        role = m.get("role", "user")
        content = m.get("content", "")

        if role == "system":
            lc_messages.append(SystemMessage(content=content))
        elif role == "assistant":
            lc_messages.append(AIMessage(content=content))
        else:  # 'user' or anything else
            lc_messages.append(HumanMessage(content=content))
    return lc_messages

## 4. Calling OpenRouter

In this section we define a small helper function that sends a request to the OpenRouter API and returns the model’s response.

We will reuse this function with different `model_id` values (GPT, Claude, Qwen) by simply changing the `model_id` argument, while keeping the rest of the code identical.


In [8]:
def call_openrouter_model(model_id, messages, temperature, top_p, max_tokens):
    """
    Send a chat completion request to OpenRouter via LangChain and return:
        (response_text, latency_seconds)
    """
    # Instantiate LangChain ChatOpenAI targeting OpenRouter
    llm = ChatOpenAI(
        model=model_id,
        api_key=OPENROUTER_API_KEY,
        base_url="https://openrouter.ai/api/v1",
        temperature=float(temperature) if temperature is not None else 0.0,
        max_tokens=max_tokens,
        top_p=float(top_p) if top_p is not None else None,
        default_headers={
            "HTTP-Referer": "https://colab.research.google.com/",
            "X-Title": "Multi-Model Project Generator",
        },
    )

    lc_messages = to_langchain_messages(messages)

    t0 = time.time()
    response = llm.invoke(lc_messages)
    latency = time.time() - t0

    content = response.content
    return content, latency


## 5. Project Titles and Descriptions:

For the first part of the dataset generation, we want to generate 500 AI-related project titles and descriptions. For that, we use the **QWEN3-Coder** LLM model.

### 5.1. System Prompt and Examples

First, we need to clearly define **the role of the model** and **how it should respond**.  
We do this by writing a **system prompt**, which acts like the model’s “rules and job description” during dataset generation.
A good system prompt helps the model stay consistent, follow our format, and avoid hallucinating outside the task.

We also provide a few **examples** so the model can imitate the exact structure and style we expect in the dataset.

In [ ]:
IDEA_EXAMPLES = """
```
{
  "title": "Iris Species Classification with Decision Trees",
  "description": "Build a decision tree classifier to predict iris species using the scikit-learn iris dataset. Train the model, evaluate its accuracy, and visualize the decision boundaries. Compare performance with different tree depths.",
  "difficulty": "easy"
},

{
  "title": "MNIST Digit Recognition with CNN",
  "description": "Create a convolutional neural network to classify handwritten digits from the MNIST dataset available in keras.datasets. Implement data augmentation, train the model, and achieve over 98% accuracy on the test set.",
  "difficulty": "medium"
},

{
  "title": "Advanced Ensemble Methods for California Housing",
  "description": "Implement and compare multiple ensemble techniques (Random Forest, Gradient Boosting, XGBoost, Stacking) for predicting house prices using scikit-learn's California housing dataset. Perform extensive hyperparameter tuning and feature engineering to optimize performance.",
  "difficulty": "hard"
}
```
""".strip()

In [ ]:
IDEA_SYSTEM_PROMPT = f"""
You are a meticulous AI project designer. Across multiple calls, your overall task is to help create a global dataset of exactly 500 distinct AI-related programming project ideas for finetuning a language model.

In EACH call, you generate ONE BATCH of exactly 50 NEW project ideas.

The user message may include a JSON array of previously accepted projects (for example under a key like "previous_projects" or similar). Those previous projects are ALREADY part of the global dataset. In the current batch, you MUST NOT repeat or slightly rephrase any of them.

====================
1. GLOBAL & BATCH REQUIREMENTS
====================
Global objective (all batches together):
- Total: 500 projects.
- Global difficulty distribution target:
  - 300 projects labeled "easy"
  - 150 projects labeled "medium"
  - 50 projects labeled "hard"

Batch objective (this call only):
- You MUST return exactly 50 project objects in this response.
- For this batch of 50:
  - 30 projects MUST be labeled "easy"
  - 15 projects MUST be labeled "medium"
  - 5 projects MUST be labeled "hard"
- Total in this batch: 30 + 15 + 5 = 50. No more, no less.

Uniqueness across batches:
- Treat all previously provided projects from the user (if any) as ALREADY USED.
- Do NOT repeat any previous title exactly.
- Do NOT create trivial variations of previous titles or descriptions (e.g., only changing a few words while keeping the same idea).

====================
2. EXECUTION CONSTRAINTS
====================
Each project must be:

- Runnable in a single Jupyter or Google Colab notebook.
- Implementable WITHOUT external files, folders, or manual file uploads.

====================
3. DATASET CONSTRAINTS
====================
All projects MUST use datasets that are directly available from standard Python libraries that can be installed via pip and imported in a notebook.

Acceptable dataset sources include, for example:
- scikit-learn built-in datasets
  (e.g., load_iris, load_digits, load_wine, fetch_california_housing, etc.)
- keras.datasets
  (e.g., mnist, cifar10, fashion_mnist, imdb, reuters, etc.)
- seaborn example datasets
  (e.g., tips, titanic, penguins, diamonds, etc.)
- statsmodels datasets
- Other similar libraries with built-in datasets.

Forbidden:
- Do NOT require downloading datasets from external URLs (Kaggle, UCI, etc.).
- Do NOT require uploading local files.

If you are unsure whether a dataset is built-in, prefer a well-known one from the list above.

====================
4. DIVERSITY AND UNIQUENESS
====================
Global diversity goal:
The 500 projects, taken together across all batches, should cover a wide range of AI/ML topics, including but not limited to:
- Classification and regression
- Clustering and dimensionality reduction
- Neural networks and deep learning
- Natural language processing
- Computer vision
- Time series analysis
- Recommendation systems
- Ensemble methods
- Model evaluation and comparison
- Feature engineering and selection
- Hyperparameter tuning

Within this batch of 50:
- Cover multiple datasets (do NOT use only one or two datasets).
- Vary algorithms and techniques, such as:
  - logistic regression, SVM, decision trees, random forests, gradient boosting, naive Bayes, KNN, XGBoost, LightGBM, etc.
  - CNNs, RNNs, LSTMs, transformers, autoencoders, GNNs, etc.
- Vary problem types:
  - binary classification, multiclass classification, regression, clustering, time series forecasting, anomaly detection, etc.
- Vary evaluation metrics:
  - accuracy, precision, recall, F1, ROC-AUC, RMSE, MAE, MAPE, etc.
- Vary preprocessing and feature engineering:
  - scaling, encoding, feature selection, feature importance, dimensionality reduction, etc.

Uniqueness rules (current batch + previous batches):
- Titles must be UNIQUE after lowercasing and trimming spaces, and must not appear in any of the previously provided projects.
- Descriptions must be clearly different in content and focus from each other and from any previous descriptions.
- Avoid “near-duplicates” where only a few words change, such as:
  - BAD: "Iris Classification with Random Forest"
  - BAD: "Iris Classification using Random Forest Classifier"
- Instead, vary algorithms, datasets, tasks, or analysis focus, e.g.:
  - GOOD: "Iris Classification with Random Forest"
  - GOOD: "Iris Species Prediction using Gradient Boosting"
  - GOOD: "Iris Feature Importance Analysis with Multiple Models"
  - GOOD: "Hyperparameter Tuning Random Forest on Iris Dataset"

====================
5. PER-PROJECT FORMAT
====================
You MUST output ONE single valid JSON array containing exactly 50 objects in this batch.
Each object MUST have exactly these three keys:

1) "title"
   - A short, clear, UNIQUE name for the project.
   - Maximum 6 words.
   - Use plain English words only.
   - MUST NOT duplicate or trivially rephrase any title in the current batch or in the list of previous projects from the user.
   - Example valid titles:
     - "Iris Species Classification"
     - "MNIST Digit Recognition CNN"
     - "Titanic Survival Prediction Model"

2) "description"
   - 2–4 sentences.
   - Clearly describe:
     - Which dataset is used (and from which library).
     - What the main ML task is.
     - Any key techniques or steps (e.g., preprocessing, model training, evaluation).
   - Each description MUST be distinct and not a trivial rephrase of another project (current batch or previous batches).

3) "difficulty"
   - Exactly one of: "easy", "medium", "hard" (all lowercase).
   - For this batch of 50, you MUST produce:
     - 30 "easy"
     - 15 "medium"
     - 5 "hard"

====================
6. EXAMPLES (DO NOT COPY)
====================
Here are a few examples of valid project objects. They are ONLY for style and format reference.
Do NOT include these example objects in the final JSON array.

{IDEA_EXAMPLES}

====================
7. OUTPUT INSTRUCTIONS (CRITICAL)
====================
Follow these output rules exactly:

- Return ONLY the raw JSON array as the entire response for this call.
- Start the response with '[' and end it with ']'.
- Do NOT wrap the JSON in ```json or any other code fences.
- Do NOT add any extra text, comments, or explanations before or after the JSON array.
- Ensure the JSON is syntactically valid:
  - Proper commas between objects.
  - Double quotes for all string keys and values.
  - Proper escaping of quotes and special characters inside strings.

Before you finish, mentally check:

- Total number of objects in THIS BATCH = 50.
- Difficulty counts in THIS BATCH:
  - 30 with "difficulty": "easy"
  - 15 with "difficulty": "medium"
  - 5 with "difficulty": "hard"
- All titles in this batch are unique (case-insensitive).
- None of the titles or descriptions repeat or trivially rephrase any of the previously provided projects from the user.
- Every object has exactly: "title", "description", "difficulty".
""".strip()


### 5.2. User prompt:

This short user prompt simply tells the model to start generating the JSON array, while the system prompt already defines all the rules and format.


In [ ]:
PREVIOUS_PROJECTS = []

In [ ]:
IDEA_USER_PROMPT= f"""
You are generating ONE BATCH of NEW project ideas to be added to a global dataset of 500 projects, following all rules in the system prompt.

Below is the list of projects that are ALREADY in the dataset from previous batches.
They are provided as a JSON array called "previous_projects".
You MUST NOT repeat any of these titles, and you MUST NOT create trivial variations of these ideas.

previous_projects:
{PREVIOUS_PROJECTS}

Your task in THIS CALL:
- Generate EXACTLY 50 NEW AI/ML programming project objects.
- None of them may duplicate or trivially rephrase any project in previous_projects.

EXACT COUNT REQUIRED FOR THIS BATCH:
- 30 projects with "difficulty": "easy"
- 15 projects with "difficulty": "medium"
- 5 projects with "difficulty": "hard"
= 50 TOTAL in this batch (no more, no less)

UNIQUENESS IS CRITICAL:
- Every project in this batch must be meaningfully different from:
  - All other projects in this batch, AND
  - All projects in previous_projects.
- Titles must be unique (case-insensitive) and contain at most 6 words.
- Vary datasets, algorithms, problem types, evaluation metrics, and analysis focus so each project teaches something distinct.

OUTPUT FORMAT (CRITICAL):
- Output ONLY a single JSON array as your entire response.
- NO markdown, NO code fences, NO explanations.
- Start immediately with `[` and end with `]`.
- Each object must have exactly these keys:
  {{ "title": "...", "description": "...", "difficulty": "easy|medium|hard" }}

Before you output, mentally verify:
1. The array length is exactly 50.
2. Difficulty counts are exactly: 30 easy, 15 medium, 5 hard.
3. No title in this batch appears in previous_projects.
4. No description in this batch is a trivial rephrase of a previous project.
5. Every object has valid "title", "description", and "difficulty" fields and follows all dataset and execution rules from the system prompt.

Now generate the NEXT 50 unique projects as a pure JSON array for this batch.
""".strip()


### 5.3 Generate project titles and descriptions

In this step we generate the AI-related project titles and descriptions using the GPT-OSS 120B large language model.
Instead of asking for all 500 projects in a single call (which often leads to hallucinations, bad JSON, or repeated ideas), we generate them in batches of 50 projects at a time.

For each batch we:

* Ask the model for 50 new projects following the system rules

* Feed in the previously accepted projects so it can avoid duplicates

* Parse and validate the JSON output

* Prune or clean any duplicate or low-quality items

By repeating this process over multiple batches, we end up with a clean, diverse, and accurate dataset of 500 projects, saved as a single JSON file.

In [ ]:
def normalize_title(title):
    """Normalize a title for duplicate checking."""
    return (title or "").strip().lower()

In [ ]:
def generate_idea_dataset(
    model_id,
    output_path="idea_dataset.json",
    raw_path="idea_dataset_raw.txt",
    N=500,                 # total number of projects to generate
    batch_size=50,         # projects per model call (assumed to be 50 here)
    temperature=0.2,
    top_p=0.9,
    max_tokens=15000,
    max_batches=30,        # safety limit to avoid infinite loops
):
    """
    Generate a full idea dataset by calling the model in batches.

    - Calls the model to generate `batch_size` projects at a time.
    - Each batch includes all previously accepted projects in the user prompt
      (via the {PREVIOUS_PROJECTS} placeholder) so the model can avoid duplicates.
    - Filters out duplicates in Python as a second safety layer.
    - Repeats until we reach N unique projects or hit max_batches.
    """

    if N % batch_size != 0:
        print(f"[warning] N={N} is not a multiple of batch_size={batch_size}. "
              f"The last batch may overshoot and be trimmed.")

    previous_projects = []
    batch_index = 1

    # Prepare base path for per-batch raw outputs
    raw_base, raw_ext = os.path.splitext(raw_path)
    if not raw_ext:
        raw_ext = ".txt"

    while len(previous_projects) < N and batch_index <= max_batches:
        print(f"\n=== Batch {batch_index} ===")
        print(f"Currently have {len(previous_projects)} projects (target: {N}).")

        # 1) Build user prompt with all previous projects injected
        prev_json = json.dumps(previous_projects, ensure_ascii=False, indent=2)
        user_prompt = IDEA_USER_PROMPT.replace("{PREVIOUS_PROJECTS}", prev_json)

        messages = [
            {"role": "system", "content": IDEA_SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ]

        print(f"Calling model: {model_id}")
        raw_text, latency = call_openrouter_model(
            model_id= model_id,
            messages=messages,
            temperature=temperature,
            top_p=top_p,
            max_tokens=max_tokens,
        )
        print(f"Latency: {latency:.2f}s")
        print(f"Raw output length: {len(raw_text)} characters")

        # 3) Save raw output for this batch
        batch_raw_path = f"{raw_base}_batch{batch_index}{raw_ext}"
        try:
            with open(batch_raw_path, "w", encoding="utf-8") as f:
                f.write(raw_text)
            print(f"Raw model output for batch {batch_index} saved to: {batch_raw_path}")
        except Exception as e:
            print(f"[warning] Failed to save raw output for batch {batch_index}: {e}")

        # 4) Extract + parse JSON array for this batch
        batch_data = extract_json_array(raw_text)
        if batch_data is None:
            print(f"[error] Could not extract a valid JSON array from batch {batch_index}.")
            print("Check the raw batch file above to see what the model actually returned.")
            batch_index += 1
            continue

        print(f"Parsed {len(batch_data)} items from batch {batch_index}. Validating...")

        # 5) Validate batch structure (should be exactly batch_size items)
        batch_errors = validate_items(batch_data, N=batch_size)
        if batch_errors:
            print(f"[validation errors in batch {batch_index}]")
            for err in batch_errors:
                print(" -", err)
            # Skip this batch and move on
            batch_index += 1
            continue

        # 6) Filter out duplicates vs previous_projects based on normalized title
        existing_titles = {normalize_title(p.get("title")) for p in previous_projects}
        unique_new_items = []
        dropped_duplicates = 0

        for item in batch_data:
            t_norm = normalize_title(item.get("title"))
            if not t_norm or t_norm in existing_titles:
                dropped_duplicates += 1
                continue
            existing_titles.add(t_norm)
            unique_new_items.append(item)

        print(
            f"Batch {batch_index}: accepted {len(unique_new_items)} new items, "
            f"dropped {dropped_duplicates} duplicates vs previous batches."
        )

        # 7) Append unique items to the global list
        previous_projects.extend(unique_new_items)
        print(f"Total projects after batch {batch_index}: {len(previous_projects)}")

        # 8) If we overshoot N, trim down
        if len(previous_projects) > N:
            previous_projects = previous_projects[:N]

        batch_index += 1

    # 9) Final check
    if len(previous_projects) < N:
        print(
            f"[warning] Finished {batch_index - 1} batches but only collected "
            f"{len(previous_projects)} unique projects (target was {N})."
        )

    # Final validation on the full dataset (only if we reached N)
    if len(previous_projects) == N:
        print("\nRunning final validation on the full dataset...")
        final_errors = validate_items(previous_projects, N=N)
        if final_errors:
            print("[final validation errors]")
            for err in final_errors:
                print(" -", err)
        else:
            print("Final dataset passed validation ✅")

    # 10) Save clean JSON
    try:
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(previous_projects, f, ensure_ascii=False, indent=2)
        print(f"Saved idea dataset to: {output_path}")
    except Exception as e:
        print(f"[error] Failed to save JSON file: {e}")
        return None

    return previous_projects


In [ ]:
ideas = generate_idea_dataset(model_id= MODELS["gpt"])


=== Batch 1 ===
Currently have 0 projects (target: 500).
Calling model: openai/gpt-oss-120b
Latency: 148.36s
Raw output length: 15471 characters
Raw model output for batch 1 saved to: idea_dataset_raw_batch1.txt
Parsed 50 items from batch 1. Validating...
Batch 1: accepted 50 new items, dropped 0 duplicates vs previous batches.
Total projects after batch 1: 50

=== Batch 2 ===
Currently have 50 projects (target: 500).
Calling model: openai/gpt-oss-120b
Latency: 147.82s
Raw output length: 15987 characters
Raw model output for batch 2 saved to: idea_dataset_raw_batch2.txt
Parsed 50 items from batch 2. Validating...
Batch 2: accepted 46 new items, dropped 4 duplicates vs previous batches.
Total projects after batch 2: 96

=== Batch 3 ===
Currently have 96 projects (target: 500).
Calling model: openai/gpt-oss-120b
Latency: 142.46s
Raw output length: 0 characters
Raw model output for batch 3 saved to: idea_dataset_raw_batch3.txt
[error] Could not extract a valid JSON array from batch 3.
Ch

#### 5.3.1. Sorting and Fixing the number of samples:

To make the dataset cleaner and easier to use, we sort all projects by difficulty and then adjust the labels so the final distribution matches our target: 300 **easy**, 150 **medium**, and 50 **hard** project topics.

In [ ]:
# 1. Load the dataset
with open("idea_dataset.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Total items in dataset: {len(data)}")

# 2. Normalize titles (case-insensitive, strip spaces)
def normalize_title(t: str) -> str:
    return (t or "").strip().lower()

titles = [normalize_title(item.get("title", "")) for item in data]

# 3. Count occurrences
counts = Counter(titles)

# 4. Extract duplicates
duplicate_titles = {title: count for title, count in counts.items() if count > 1}

print(f"Number of distinct duplicated titles: {len(duplicate_titles)}")
print(f"Total extra items due to duplication: {sum(c - 1 for c in duplicate_titles.values())}")
print()

# 5. Show some examples
if duplicate_titles:
    print("Duplicated titles (normalized) and their counts:")
    for title, count in sorted(duplicate_titles.items(), key=lambda x: -x[1])[:50]:
        print(f"{count}×  {title}")
else:
    print("No duplicate titles found ✅")


Total items in dataset: 500
Number of distinct duplicated titles: 0
Total extra items due to duplication: 0

No duplicate titles found ✅


In [ ]:
medium_to_easy = [39,40,41,43,77,80,86,125,130,181,254,257,33,34,38,90,131,300,470]
hard_to_medium = [472,473,475]

for i in medium_to_easy:
    data[i]["difficulty"] = "easy"

for i in hard_to_medium:
    data[i]["difficulty"] = "medium"

In [ ]:
difficulty_order = {"easy": 0, "medium": 1, "hard": 2}

# This will sort: all easy first, then medium, then hard.
# Inside each difficulty group it also sorts alphabetically by title.
data_sorted = sorted(
    data,
    key=lambda x: (difficulty_order.get(x["difficulty"], 999), x["title"])
)

# If you want to overwrite the original list:
data[:] = data_sorted

# Then save it:
import json
with open("idea_dataset_sorted.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)


#### 5.3.2 Generating 100 additional samples with a different style

To increase the variety of projects (and later, the variety of code the model will generate), we asked the language model to produce 100 new project ideas with a slightly different focus and style. These extra samples emphasize alternative evaluation strategies, testing setups, and modeling approaches compared to the original 500.

After generating this batch, we cleaned and validated the projects (fixing labels, removing any overlaps), and then merged them with the original dataset. The final result is a 600-project dataset that is more diverse, richer in structure, and better suited for finetuning a smaller LLM.



In [ ]:
IDEA_SYSTEM_PROMPT_EXTRA = """
You are a meticulous AI project designer. Your task is to generate 100 NEW, distinct AI-related programming project ideas that extend an existing dataset of 500 projects.

These 100 projects must:

- Follow the same JSON schema as before: each project has "title", "description", "difficulty".
- Be runnable in a single Jupyter or Google Colab notebook.
- Use only datasets that are directly available via standard Python libraries
  (scikit-learn, keras.datasets, seaborn, statsmodels, or similar built-in/synthetic datasets).
- NOT duplicate or trivially rephrase any project already provided in previous_projects
  (titles or descriptions).

====================
1. FOCUS OF THESE EXTRA PROJECTS
====================
Compared to the original 500, these 100 projects should focus more on:

- Testing and evaluation:
  - writing unit tests for ML pipelines
  - building evaluation harnesses
  - defining pass/fail criteria (e.g. “model must reach at least X accuracy / RMSE below Y”)
- Robustness and reliability:
  - adversarial or noisy data tests
  - robustness to distribution shift
  - cross-validation strategies, ablation studies
- Explainability and diagnostics:
  - feature importance, SHAP / LIME analysis
  - error analysis and confusion matrix inspection
- MLOps-like aspects that still fit in a single notebook:
  - simple experiment tracking
  - model comparison dashboards
  - reproducible training pipelines with seeding and logging

Avoid simple “train one model and report accuracy” projects that are already well represented.
Each new project should add a different *evaluation or testing angle*.

====================
2. DIFFICULTY DISTRIBUTION FOR THESE 100 PROJECTS
====================
For these 100 NEW projects, use:

- 50 projects labeled "easy"
- 35 projects labeled "medium"
- 15 projects labeled "hard"

Total = 100 projects. No more, no less.

====================
3. DATASET CONSTRAINTS
====================
All projects MUST use datasets that are directly available from standard Python libraries that can be installed via pip and imported in a notebook.

Acceptable dataset sources include, for example:
- scikit-learn built-in datasets
  (e.g., load_iris, load_digits, load_wine, fetch_california_housing, etc.)
- keras.datasets
  (e.g., mnist, cifar10, fashion_mnist, imdb, reuters, etc.)
- seaborn example datasets
  (e.g., tips, titanic, penguins, diamonds, etc.)
- statsmodels datasets
- Other similar libraries with built-in datasets.

Forbidden:
- Do NOT require downloading datasets from external URLs (Kaggle, UCI, etc.).
- Do NOT require uploading local files.

If you are unsure whether a dataset is built-in, prefer a well-known one from the list above.


====================
4. DIVERSITY AND UNIQUENESS
====================
- Titles must be UNIQUE after lowercasing and trimming spaces.
- Descriptions must be clearly different from each other and from all previous_projects.
- Vary:
  - datasets used
  - algorithms and techniques
  - testing/evaluation strategies
  - pass/fail thresholds or metrics

If you reuse a dataset that appears in previous_projects, you MUST focus on a clearly different
testing, evaluation, or analysis perspective.

====================
5. PER-PROJECT FORMAT
====================
Each project object MUST have exactly these three keys:

1) "title"
   - Short, clear, UNIQUE name.
   - Maximum 6 words.
   - Plain English only.

2) "description"
   - 2–4 sentences.
   - Clearly state:
     - dataset + library,
     - main ML task,
     - specific testing / evaluation / robustness / analysis goals.

3) "difficulty"
   - Exactly one of: "easy", "medium", "hard" (lowercase).

====================
6. OUTPUT INSTRUCTIONS
====================
- Return ONLY a single JSON array with exactly 100 objects.
- NO markdown, NO code fences, NO explanations.
- Start with [ and end with ].
- Ensure valid JSON (proper commas, quotes, and escaping).
""".strip()


In [ ]:
IDEA_USER_PROMPT_EXTRA_TEMPLATE = """
Below is the JSON array previous_projects containing ALL 500 projects
that already exist in the dataset. You must NOT repeat or trivially
rephrase any of them.

previous_projects:
{PREVIOUS_PROJECTS}

Now generate the NEXT 100 NEW projects, following all rules in the system
prompt and focusing on testing, evaluation, robustness, and diagnostics.

Remember:
- EXACT counts for this batch: 50 easy, 35 medium, 15 hard.
- Titles must be unique and at most 6 words.
- Output ONLY a pure JSON array of 100 objects:
  [ {{ "title": "...", "description": "...", "difficulty": "easy|medium|hard" }}, ... ]
""".strip()

In [ ]:
# --- 1. Load existing 500-project dataset ---
with open("idea_dataset_sorted.json", "r", encoding="utf-8") as f:
    previous_projects = json.load(f)

print(f"Loaded {len(previous_projects)} existing projects from idea_dataset.json")

# --- 2. Build user prompt with previous projects injected ---
prev_json = json.dumps(previous_projects, ensure_ascii=False, indent=2)
user_prompt = IDEA_USER_PROMPT_EXTRA_TEMPLATE.replace("{PREVIOUS_PROJECTS}", prev_json)

messages = [
    {"role": "system", "content": IDEA_SYSTEM_PROMPT_EXTRA},
    {"role": "user", "content": user_prompt},
]

# --- 3. Call the model ---
model_id = MODELS["gpt"]
print(f"Calling model: {model_id}")

raw_text, latency = call_openrouter_model(
    model_id=model_id,
    messages=messages,
    temperature=0.2,
    top_p=0.9,
    max_tokens=10000,
)

print(f"Latency: {latency:.2f}s")
print(f"Raw output length: {len(raw_text)} characters")

# --- 4. Save raw model output (for debugging / recovery) ---
raw_path = "idea_dataset_extra_raw.txt"
try:
    with open(raw_path, "w", encoding="utf-8") as f:
        f.write(raw_text)
    print(f"Raw model output saved to: {raw_path}")
except Exception as e:
    print(f"[warning] Failed to save raw output: {e}")

# --- 5. Extract + parse JSON array from the model output ---
extra_data = extract_json_array(raw_text)
if extra_data is None:
    print("[error] Could not extract a valid JSON array from the model output.")
    print("Check idea_dataset_extra_raw.txt to inspect what the model returned.")
else:
    print(f"Parsed {len(extra_data)} items from model output.")

    # --- 6. Validate structure & difficulty distribution for these 100 items ---
    # This checks: correct keys, types, min/max length, difficulty in {easy, medium, hard}, and count = 100
    N_EXTRA = 100
    errs = validate_items(extra_data, N_EXTRA)
    if errs:
        print("[validation errors in extra batch]")
        for err in errs:
            print(" -", err)
    else:
        print("Extra batch passed basic validation ✅")

    # --- 7. Remove duplicates vs existing 500 (by title, case-insensitive) ---
    def norm_title(t):
        return (t or "").strip().lower()

    existing_titles = {norm_title(p.get("title")) for p in previous_projects}
    unique_extra = []
    dropped_duplicates = 0

    for item in extra_data:
        t_norm = norm_title(item.get("title"))
        if not t_norm or t_norm in existing_titles:
            dropped_duplicates += 1
            continue
        existing_titles.add(t_norm)
        unique_extra.append(item)

    print(
        f"Extra batch: kept {len(unique_extra)} unique new items, "
        f"dropped {dropped_duplicates} duplicates vs existing dataset."
    )

    # --- 8. Save the 100 (or fewer) new items as a separate file ---
    extra_output_path = "idea_dataset_extra_100.json"
    try:
        with open(extra_output_path, "w", encoding="utf-8") as f:
            json.dump(unique_extra, f, ensure_ascii=False, indent=2)
        print(f"Saved extra idea batch to: {extra_output_path}")
    except Exception as e:
        print(f"[error] Failed to save extra JSON file: {e}")

    # --- 9. Optionally: create a merged 600-project dataset ---
    merged = previous_projects + unique_extra
    merged_output_path = "idea_dataset_complete.json"
    try:
        with open(merged_output_path, "w", encoding="utf-8") as f:
            json.dump(merged, f, ensure_ascii=False, indent=2)
        print(f"Saved merged dataset (existing + extra) to: {merged_output_path}")
    except Exception as e:
        print(f"[warning] Failed to save merged dataset: {e}")


Loaded 500 existing projects from idea_dataset.json
Calling model: openai/gpt-oss-120b
Latency: 118.51s
Raw output length: 37735 characters
Raw model output saved to: idea_dataset_extra_raw.txt
Parsed 115 items from model output.
[validation errors in extra batch]
 - [{'title': 'Iris Model Unit Tests', 'description': 'Load the iris dataset from scikit-learn and train a logistic regression classifier. Write a pytest suite that checks model accuracy, predicts class probabilities, and validates that the confusion matrix contains no negative values. Ensure reproducibility with a fixed random seed.', 'difficulty': 'easy'}, {'title': 'Wine Classification Pass Fail', 'description': 'Fetch the wine dataset via scikit-learn and build a decision tree classifier. Define a pass/fail criterion that the test accuracy must exceed 85%. Automate the check and log the result in a simple CSV file.', 'difficulty': 'easy'}, {'title': 'Breast Cancer Confusion Automation', 'description': 'Load the breast_can

In [ ]:
with open("idea_dataset_extra_100.json", "r", encoding="utf-8") as f:
    extra = json.load(f)

remove_titles = [
    "Wine Bayesian Optimization Tracking",
    "Diabetes LSTM Time Series Forecast",
    "Tips Prophet Revenue Forecast",
    "CIFAR10 ResNet Hyperparameter Optimization",
    "California Stacked Ensemble Tracking",
    "MNIST CNN Early Stopping Scheduler",
    "MNIST VAE Digit Generation",
    "Sunspots SARIMAX Seasonal Forecast",
    "Sunspots LSTM Time Series Forecast",
    "California Housing Time Series Forecast",
    "Wine Stacking Bayesian Optimization",
    "XGBoost Wine Bayesian Optimization",
]

relabeled_title = "Synthetic Anomaly Isolation Forest Study"

cleaned = []
for item in extra:
    if item["title"] in remove_titles:
        continue
    if item["title"] == relabeled_title:
        item = item.copy()
        item["difficulty"] = "easy"
    cleaned.append(item)

print("len:", len(cleaned), Counter(x["difficulty"] for x in cleaned))
with open("idea_dataset_extra_100_fixed.json", "w", encoding="utf-8") as f:
    json.dump(cleaned, f, ensure_ascii=False, indent=2)


len: 100 Counter({'easy': 50, 'medium': 35, 'hard': 15})


In [ ]:
with open("idea_dataset_sorted.json", "r", encoding="utf-8") as f:
    previous_projects = json.load(f)

with open("idea_dataset_extra_100_fixed.json", "r", encoding="utf-8") as f:
    extra_fixed = json.load(f)

merged = previous_projects + extra_fixed
merged_output_path = "idea_dataset_complete_fixed.json"

with open(merged_output_path, "w", encoding="utf-8") as f:
    json.dump(merged, f, ensure_ascii=False, indent=2)

In [ ]:
with open("idea_dataset_complete_fixed.json", "r", encoding="utf-8") as f:
    all_projects = json.load(f)

difficulty_order = {"easy": 0, "medium": 1, "hard": 2}

# This will sort: all easy first, then medium, then hard.
# Inside each difficulty group it also sorts alphabetically by title.
data_sorted = sorted(
    all_projects,
    key=lambda x: (difficulty_order.get(x["difficulty"], 999), x["title"])
)

# If you want to overwrite the original list:
all_projects[:] = data_sorted

# Then save it:
with open("idea_dataset_complete_fixed_sorted.json", "w", encoding="utf-8") as f:
    json.dump(all_projects, f, ensure_ascii=False, indent=2)

## 6. The Project's correct codes:

So far, we’ve generated 600 project topics for our dataset.
The next step is to use a language model to generate the **correct**, **runnable** code for each project and attach it to the corresponding title and description.

By adding these correct solutions, the smaller model we’ll fine-tune later can learn:

* how to translate a natural-language project description into working code

* what a correct solution looks like for each task

This makes it much easier for the fine-tuned model to **check**, **debug**, and **autocorrect** new code in similar projects.

In [ ]:
with open("idea_dataset_complete_fixed_sorted.json", "r", encoding="utf-8") as f:
    dataset = json.load(f)

print(dataset[5])

{'title': 'Boston Housing Lasso Regression', 'description': 'Apply Lasso regression to the Boston housing dataset from scikit-learn to perform feature selection while predicting median value. Tune the regularization parameter via cross‑validation, report RMSE, and list selected features.', 'difficulty': 'easy'}


### 6.1. System Prompt and Examples:

In this section, we define the system prompt for the language model that will generate the correct, runnable code for each project in our dataset, based on its title and description.

We also provide a couple of few-shot examples (project → code) so the model can better understand the expected coding style, structure, and level of detail before generating solutions for the remaining projects.

In [ ]:
CORRECT_CODE_EXAMPLES = """
### Example 1:
{
  "title": "Iris Logistic Regression Classifier",
  "description": "Build a logistic regression classifier to predict iris species using the scikit-learn iris dataset. Split the data into training and test sets, standardize the features, train the model, and evaluate it using accuracy and a confusion matrix. Visualize the decision boundaries in a 2D projection to help students understand how the classifier separates classes.",
  "difficulty": "easy",
  "correct_code": "import numpy as np\nimport matplotlib.pyplot as plt\nfrom sklearn.datasets import load_iris\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay\nfrom sklearn.decomposition import PCA\n\n\ndef load_and_prepare_data():\n    \"\"\"Load the iris dataset and return features and labels.\"\"\"\n    iris = load_iris()\n    X = iris.data  # shape (150, 4)\n    y = iris.target  # 0, 1, 2\n    feature_names = iris.feature_names\n    target_names = iris.target_names\n    return X, y, feature_names, target_names\n\n\ndef train_test_split_scaled(X, y, test_size=0.2, random_state=42):\n    \"\"\"Split data into train/test sets and apply standard scaling.\"\"\"\n    X_train, X_test, y_train, y_test = train_test_split(\n        X, y, test_size=test_size, random_state=random_state, stratify=y\n    )\n    scaler = StandardScaler()\n    X_train_scaled = scaler.fit_transform(X_train)\n    X_test_scaled = scaler.transform(X_test)\n    return X_train_scaled, X_test_scaled, y_train, y_test, scaler\n\n\ndef train_logistic_regression(X_train, y_train, random_state=42):\n    \"\"\"Train a multinomial logistic regression classifier.\"\"\"\n    clf = LogisticRegression(\n        multi_class='multinomial',\n        solver='lbfgs',\n        max_iter=1000,\n        random_state=random_state\n    )\n    clf.fit(X_train, y_train)\n    return clf\n\n\ndef evaluate_model(clf, X_test, y_test, target_names):\n    \"\"\"Evaluate the classifier and plot a confusion matrix.\"\"\"\n    y_pred = clf.predict(X_test)\n    acc = accuracy_score(y_test, y_pred)\n    print(f'Accuracy on test set: {acc:.3f}')\n\n    cm = confusion_matrix(y_test, y_pred)\n    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)\n    disp.plot(cmap='Blues')\n    plt.title('Confusion Matrix - Iris Logistic Regression')\n    plt.tight_layout()\n    plt.show()\n\n\ndef plot_decision_boundaries_2d(clf, X, y, target_names):\n    \"\"\"Project data to 2D with PCA and plot decision regions.\n\n    Note: The model is trained in 4D space; we visualize decision regions by\n    projecting to 2D using PCA for educational purposes.\n    \"\"\"\n    pca = PCA(n_components=2)\n    X_2d = pca.fit_transform(X)\n\n    # Create a mesh grid over the 2D PCA space\n    x_min, x_max = X_2d[:, 0].min() - 1.0, X_2d[:, 0].max() + 1.0\n    y_min, y_max = X_2d[:, 1].min() - 1.0, X_2d[:, 1].max() + 1.0\n    xx, yy = np.meshgrid(\n        np.linspace(x_min, x_max, 200),\n        np.linspace(y_min, y_max, 200)\n    )\n\n    # Inverse transform grid back to original 4D space approximately\n    grid_2d = np.c_[xx.ravel(), yy.ravel()]\n    grid_4d = pca.inverse_transform(grid_2d)\n\n    Z = clf.predict(grid_4d)\n    Z = Z.reshape(xx.shape)\n\n    plt.figure(figsize=(8, 6))\n    plt.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.Set3)\n\n    # Scatter original 2D data\n    for i, name in enumerate(target_names):\n        plt.scatter(\n            X_2d[y == i, 0],\n            X_2d[y == i, 1],\n            label=name\n        )\n\n    plt.xlabel('PCA Component 1')\n    plt.ylabel('PCA Component 2')\n    plt.title('Iris Dataset - PCA Projection with Decision Regions')\n    plt.legend()\n    plt.tight_layout()\n    plt.show()\n\n\nif __name__ == '__main__':\n    # Reproducibility\n    np.random.seed(42)\n\n    # 1. Load data\n    X, y, feature_names, target_names = load_and_prepare_data()\n    print('Features:', feature_names)\n    print('Targets:', target_names)\n\n    # 2. Split and scale\n    X_train_scaled, X_test_scaled, y_train, y_test, scaler = train_test_split_scaled(X, y)\n\n    # 3. Train model\n    clf = train_logistic_regression(X_train_scaled, y_train)\n\n    # 4. Evaluate model\n    evaluate_model(clf, X_test_scaled, y_test, target_names)\n\n    # 5. Visualize 2D decision regions (optional but educational)\n    plot_decision_boundaries_2d(clf, X, y, target_names)\n"
}
### Example 2:
{
  "title": "IMDB Sentiment LSTM Classifier",
  "description": "Train a sentiment analysis model on the IMDB movie reviews dataset using an LSTM neural network. Load and preprocess the dataset from keras.datasets, including tokenization and sequence padding. Build, train, and evaluate the LSTM model, then report accuracy and visualize the training and validation loss curves.",
  "difficulty": "medium",
  "correct_code": "import numpy as np\nimport matplotlib.pyplot as plt\nimport tensorflow as tf\nfrom tensorflow import keras\nfrom tensorflow.keras import layers\n\n\ndef load_imdb_data(num_words=10000, maxlen=200):\n    \"\"\"Load the IMDB dataset and preprocess it (pad/truncate sequences).\"\"\"\n    (x_train, y_train), (x_test, y_test) = keras.datasets.imdb.load_data(num_words=num_words)\n\n    x_train = keras.preprocessing.sequence.pad_sequences(x_train, maxlen=maxlen)\n    x_test = keras.preprocessing.sequence.pad_sequences(x_test, maxlen=maxlen)\n\n    return (x_train, y_train), (x_test, y_test)\n\n\ndef build_lstm_model(num_words=10000, embedding_dim=64):\n    \"\"\"Build a simple LSTM-based sentiment classifier.\"\"\"\n    model = keras.Sequential([\n        layers.Embedding(input_dim=num_words, output_dim=embedding_dim),\n        layers.LSTM(64),\n        layers.Dense(1, activation='sigmoid')\n    ])\n\n    model.compile(\n        loss='binary_crossentropy',\n        optimizer='adam',\n        metrics=['accuracy']\n    )\n    return model\n\n\ndef plot_training_history(history):\n    \"\"\"Plot training and validation loss and accuracy curves.\"\"\"\n    acc = history.history.get('accuracy', [])\n    val_acc = history.history.get('val_accuracy', [])\n    loss = history.history.get('loss', [])\n    val_loss = history.history.get('val_loss', [])\n\n    epochs = range(1, len(acc) + 1)\n\n    plt.figure(figsize=(12, 5))\n\n    plt.subplot(1, 2, 1)\n    plt.plot(epochs, acc, 'b-', label='Training acc')\n    plt.plot(epochs, val_acc, 'r--', label='Validation acc')\n    plt.xlabel('Epochs')\n    plt.ylabel('Accuracy')\n    plt.title('Training and Validation Accuracy')\n    plt.legend()\n\n    plt.subplot(1, 2, 2)\n    plt.plot(epochs, loss, 'b-', label='Training loss')\n    plt.plot(epochs, val_loss, 'r--', label='Validation loss')\n    plt.xlabel('Epochs')\n    plt.ylabel('Loss')\n    plt.title('Training and Validation Loss')\n    plt.legend()\n\n    plt.tight_layout()\n    plt.show()\n\n\nif __name__ == '__main__':\n    # Reproducibility\n    np.random.seed(42)\n    tf.random.set_seed(42)\n\n    # Hyperparameters\n    NUM_WORDS = 10000\n    MAXLEN = 200\n    EMBEDDING_DIM = 64\n    BATCH_SIZE = 64\n    EPOCHS = 5\n\n    # 1. Load and preprocess data\n    print('Loading and preprocessing IMDB dataset...')\n    (x_train, y_train), (x_test, y_test) = load_imdb_data(num_words=NUM_WORDS, maxlen=MAXLEN)\n    print('Training samples:', x_train.shape[0])\n    print('Test samples:', x_test.shape[0])\n\n    # 2. Build model\n    model = build_lstm_model(num_words=NUM_WORDS, embedding_dim=EMBEDDING_DIM)\n    model.summary()\n\n    # 3. Train model with validation split\n    history = model.fit(\n        x_train,\n        y_train,\n        epochs=EPOCHS,\n        batch_size=BATCH_SIZE,\n        validation_split=0.2,\n        verbose=1\n    )\n\n    # 4. Evaluate on test set\n    test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)\n    print(f'Test accuracy: {test_acc:.3f}')\n\n    # 5. Plot training history\n    plot_training_history(history)\n"
}
""".strip()


In [ ]:
CORRECT_CODE_SYSTEM_PROMPT = f"""
You are an expert Python programmer and AI educator. Your task is to generate complete, correct, and executable Python code for an AI project based on a given title, description, and difficulty. The code should be suitable for students and beginners in AI and data science.

You will receive a single project (title, description, difficulty) from the dataset in the user message. Your job is to write the corresponding Python implementation that correctly fulfills the project description, using the given difficulty level as a guide for complexity.

====================
Code requirements
====================
The code you generate must:

- Be written in Python.
- Be fully runnable in a single Jupyter or Google Colab notebook cell / script.
- Include all necessary imports at the top.
- Only use the Python standard library and these common ML/AI libraries when necessary: numpy, pandas, matplotlib, scikit-learn, scipy, statsmodels, tensorflow/keras, torch; do not import any library you don’t actually use, and do not include unused imports.
- If the project uses a dataset, prefer:
  - built-in datasets from scikit-learn, keras.datasets, statsmodels, OR
  - synthetic datasets generated with numpy / scikit-learn (e.g., make_classification, make_blobs).
- Do NOT require downloading external files (Kaggle, arbitrary URLs) or manual file uploads.
- Be clear, well-structured, and well-commented, especially around key ML steps (data loading, preprocessing, model creation, training, evaluation, visualization).
- Use descriptive variable and function names following Python conventions.
- Include a short “main flow” or example usage at the end, so running the script will demonstrate the model’s behavior.
- Handle common edge cases where reasonable (e.g., missing values) with simple checks or comments.
- Do not include unused imports. Only import libraries that are actually referenced in the code. If you don’t use a library, remove its import line.

- Reproducibility: use fixed seeds where applicable (e.g., np.random.seed(42)) and set random_state=42 for scikit-learn models and train_test_split when supported. If using TensorFlow/Keras or torch, also set their random seeds where applicable.

- Runtime budget: keep runtime reasonable for a notebook environment (use modest model sizes and training time). For deep learning tasks, prefer small architectures and a small number of epochs (typically 3–10 unless the description explicitly requires more), and consider using a subset of data if needed to keep runtime short.

- Avoid flaky performance assertions: do NOT assert thresholds on model performance metrics (e.g., accuracy, R², MAE, p-values, FNR) on real-world datasets, since they can vary across environments. If an assert is requested by the project, prefer asserting deterministic properties (e.g., array shapes, non-empty outputs, file existence) unless the description explicitly demands a performance threshold.

====================
Important dataset rule (Boston replacement)
====================
The scikit-learn function `load_boston()` is deprecated/removed in modern versions of scikit-learn.

Therefore:
- If (and ONLY if) the input project title or description refers to the Boston housing dataset from scikit-learn (e.g., "Boston housing", "load_boston", "Boston dataset from scikit-learn"),
  you MUST adapt the project to use the California housing dataset instead via:
    `from sklearn.datasets import fetch_california_housing`
  and you MUST update the output "title" and "description" accordingly so they accurately describe the California housing dataset.

- In all other cases (i.e., when the project is NOT about Boston/load_boston), you MUST NOT change the title, description, or difficulty. They are immutable.

====================
Planning (scratchpad)
====================
Before writing the final code, think through your approach in an internal scratchpad:

<scratchpad>
1. What are the main components needed for this project?
2. What libraries/imports are required?
3. What functions or classes should be created?
4. What is the logical flow of the code (data → preprocessing → model → training → evaluation → visualization)?
5. Are there any potential issues or edge cases to handle?
6. Final import audit: Which imports did I include? Is each one actually used in the code?
7. Reproducibility check: Did I set random seeds / random_state where applicable?
8. Runtime check: Is training time modest (especially epochs/model size)?
9. Assertion check: Did I avoid asserting unstable performance thresholds on real data unless explicitly required?
</scratchpad>

Do NOT include the scratchpad in your final output. It is only for your internal reasoning.

====================
Examples
====================
You are also given a few example projects with their correct solutions to illustrate the desired style and format:

<examples>
{{CORRECT_CODE_EXAMPLES}}
</examples>

Use these examples only as guidance for structure, commenting style, and level of detail. Do NOT copy the example code. For each new project you receive, you must produce a fresh, fully working solution.

====================
Output format (STRICT)
====================
Your ENTIRE response must be a single valid JSON object with exactly these four keys and nothing else:

- "title":
  - MUST be exactly the same string as in the input project ({{PROJECT_TITLE}}),
    EXCEPT the Boston replacement rule above (then update it to match California housing).
- "description":
  - MUST be exactly the same string as in the input project ({{PROJECT_DESCRIPTION}}),
    EXCEPT the Boston replacement rule above (then update it to match California housing).
- "difficulty":
  - MUST be exactly the same string as in the input project ({{PROJECT_DIFFICULTY}}), without any change.
- "correct_code":
  - A single string containing the complete, executable Python code.

Additional constraints for the JSON:

- Do NOT add any extra keys.
- Do NOT wrap the JSON in markdown, code fences, XML tags, or any other text.
- The "correct_code" field must be a single JSON string:
  - Use \\n for line breaks.
  - Use spaces for indentation.
  - Escape internal double quotes (") and backslashes (\\\\) correctly.
  - Do NOT include backticks or markdown code fences inside the string.

The final response MUST be valid JSON that can be parsed directly, with exactly these four fields:
"title", "description", "difficulty", "correct_code".
""".strip()


### 6.2. User Prompt:

Now that we’ve defined the system prompt and provided two few-shot examples, we can build the user prompt.
The user prompt will simply pass one project from our dataset (its title, description, and difficulty) to the model and ask it to generate the corresponding correct Python code.

In [ ]:
CORRECT_CODE_USER_PROMPT = """
Using the rules and style described in the system message and the examples, generate the complete, correct Python implementation for this project.

<project_input>
{
  "title": "{{PROJECT_TITLE}}",
  "description": "{{PROJECT_DESCRIPTION}}",
  "difficulty": "{{PROJECT_DIFFICULTY}}"
}
</project_input>

Remember:
- Do NOT modify the title, description, or difficulty.
- Return ONLY a single JSON object with exactly these four keys:
  - "title"
  - "description"
  - "difficulty"
  - "correct_code"

The first three fields must exactly match the values in <project_input>.
The "correct_code" field must contain the full runnable Python code as a single JSON string (with \\n for line breaks, properly escaped).
Do not include any extra text, comments, or markdown outside of the JSON object.
""".strip()


### 6.3. Generating correct codes:

We now need to generate the correct Python code for all 600 projects.
Because this is a large number of samples, we won’t send them to the model all at once. Instead, we:

* process the projects one by one,

* use a function that takes a start index and end index so we can work in small batches (e.g. 50 projects at a time),

* inspect and sanity-check the generated code as we go (to catch obvious errors early).

In this step, we implement a helper function that loops over a slice of the dataset, calls the language model for each project, and stores the returned `correct_code` in our dataset.

In [ ]:
def extract_json_object(text: str, return_candidate_on_fail: bool = False):
    """
    Try to extract and parse a single JSON object from model output.

    Handles:
    - ```json ... ``` fences
    - Optional <json_output>...</json_output> tags
    - Extra junk before/after the JSON (takes first '{' to last '}')

    Returns:
        - dict on success
        - None if nothing plausible found and return_candidate_on_fail=False
        - str (candidate JSON slice) if parsing fails and return_candidate_on_fail=True
    """
    original = text

    # 1) Strip ```json ... ``` fences if present
    m = re.search(r"```(?:json)?\s*([\s\S]*?)\s*```", text, re.IGNORECASE)
    if m:
        text = m.group(1).strip()

    # 2) Strip <json_output>...</json_output> if present
    if "<json_output>" in text and "</json_output>" in text:
        start_tag = text.index("<json_output>") + len("<json_output>")
        end_tag = text.rindex("</json_output>")
        text = text[start_tag:end_tag].strip()

    # 3) Try direct parse first
    try:
        parsed = json.loads(text)
        if isinstance(parsed, dict):
            return parsed
    except Exception:
        pass

    # 4) Fallback: take from first '{' to last '}' as candidate object
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        if return_candidate_on_fail:
            return text
        return None

    candidate = text[start:end + 1].strip()

    try:
        parsed = json.loads(candidate)
        if isinstance(parsed, dict):
            return parsed
    except Exception as e:
        if return_candidate_on_fail:
            print(f"[extract_json_object] JSON parsing failed: {e}")
            return candidate
        return None


In [ ]:
def _is_boston_project(title: str, desc: str) -> bool:
    t = (title or "").lower()
    d = (desc or "").lower()
    return ("boston" in t) or ("boston" in d) or ("load_boston" in t) or ("load_boston" in d)

def _rewrite_boston_to_california(text: str) -> str:
    if not text:
        return text
    s = text
    # Replace dataset naming
    s = re.sub(r"\bBoston\b", "California", s, flags=re.IGNORECASE)
    s = re.sub(r"\bload_boston\b", "fetch_california_housing", s, flags=re.IGNORECASE)
    # Slightly clarify if it used sklearn wording
    if "california" in s.lower() and "fetch_california_housing" not in s:
        s = s.replace("California housing dataset", "California housing dataset (fetch_california_housing)")
    return s


In [ ]:
def generate_correct_code(
    model_id,
    dataset,
    start_idx=0,
    end_idx=None,
    system_prompt=None,
    user_template=None,
    temperature=0.2,
    top_p=0.9,
    max_tokens=10000,
    raw_dir="correct_code_raw",
    codes_output_path= None,
    sleep_between=0.5,
):
    """
    Generate correct_code for each project in [start_idx, end_idx) using a chat model.

    IMPORTANT: This function does NOT modify the original dataset.
    It only reads from `dataset` and writes results to a separate JSON file.

    Boston special-case:
    - If the input mentions Boston/load_boston, we ALLOW the model to rewrite title/description to California.
    - Otherwise, title/description/difficulty must remain exactly unchanged.
    """

    # Resolve prompts
    if system_prompt is None:
        try:
            system_prompt = CORRECT_CODE_SYSTEM_PROMPT
        except NameError:
            raise ValueError("system_prompt is None and CORRECT_CODE_SYSTEM_PROMPT is not defined.")

    if user_template is None:
        try:
            user_template = CORRECT_CODE_USER_PROMPT
        except NameError:
            raise ValueError("user_template is None and CORRECT_CODE_USER_PROMPT is not defined.")

    # Index range
    if end_idx is None:
        end_idx = len(dataset)
    if start_idx < 0 or end_idx > len(dataset) or start_idx >= end_idx:
        raise ValueError(f"Invalid start/end range: start_idx={start_idx}, end_idx={end_idx}, len={len(dataset)}")

    os.makedirs(raw_dir, exist_ok=True)

    # Load or init output file
    if codes_output_path and os.path.exists(codes_output_path):
        try:
            with open(codes_output_path, "r", encoding="utf-8") as f:
                codes_list = json.load(f)
            if not isinstance(codes_list, list):
                print(f"[warning] {codes_output_path} did not contain a list. Overwriting.")
                codes_list = []
        except Exception as e:
            print(f"[warning] Failed to read existing {codes_output_path}: {e}")
            codes_list = []
    else:
        codes_list = []

    total = end_idx - start_idx
    print(f"Generating correct_code for items [{start_idx}, {end_idx}) (total {total}) using model: {model_id}")
    print(f"Results will be appended to: {codes_output_path}")

    for idx in range(start_idx, end_idx):
        proj = dataset[idx]

        orig_title = (proj.get("title") or "").strip()
        orig_desc  = (proj.get("description") or "").strip()
        orig_diff  = (proj.get("difficulty") or "").strip()

        if not orig_title or not orig_desc or not orig_diff:
            print(f"[warning] Item {idx} is missing title/description/difficulty. Skipping.")
            continue

        boston_case = _is_boston_project(orig_title, orig_desc)

        print(f"\n=== Project {idx} / {end_idx - 1} ===")
        print(f"Title: {orig_title}")
        print(f"Difficulty: {orig_diff}")
        if boston_case:
            print("[info] Boston case detected → California rewrite is allowed for title/description.")

        # Build user prompt from template
        user_prompt = (
            user_template
            .replace("{{PROJECT_TITLE}}", orig_title)
            .replace("{{PROJECT_DESCRIPTION}}", orig_desc)
            .replace("{{PROJECT_DIFFICULTY}}", orig_diff)
        )

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]

        # Call OpenRouter
        raw_text, latency = call_openrouter_model(
            model_id=model_id,
            messages=messages,
            temperature=temperature,
            top_p=top_p,
            max_tokens=max_tokens,
        )
        print(f"Latency: {latency:.2f}s | Raw length: {len(raw_text)} chars")

        # Save raw response for debugging
        raw_path = os.path.join(raw_dir, f"project_{idx:04d}.txt")
        try:
            with open(raw_path, "w", encoding="utf-8") as f:
                f.write(raw_text)
        except Exception as e:
            print(f"[warning] Failed to save raw output for item {idx}: {e}")

        # Parse JSON object
        parsed = extract_json_object(raw_text, return_candidate_on_fail=False)
        if parsed is None:
            print(f"[error] Could not extract a valid JSON object for item {idx}.")
            print(f"Check raw file: {raw_path}")
            continue

        expected_keys = {"title", "description", "difficulty", "correct_code"}
        missing = expected_keys - set(parsed.keys())
        extra = set(parsed.keys()) - expected_keys
        if missing:
            print(f"[warning] Item {idx}: missing keys in model output: {missing}")
        if extra:
            print(f"[warning] Item {idx}: extra keys in model output: {extra}")

        parsed_title = (parsed.get("title") or "").strip()
        parsed_desc  = (parsed.get("description") or "").strip()
        parsed_diff  = (parsed.get("difficulty") or "").strip()
        correct_code = parsed.get("correct_code", "")

        # Difficulty must never change
        if parsed_diff and parsed_diff != orig_diff:
            print(f"[warning] Item {idx}: model changed difficulty (NOT allowed).")
            print(f"  expected: {orig_diff}")
            print(f"  got:      {parsed_diff}")

        # Decide what we SAVE
        if boston_case:
            # Allowed: title/description may change to California
            final_title = parsed_title if parsed_title else _rewrite_boston_to_california(orig_title)
            final_desc  = parsed_desc  if parsed_desc  else _rewrite_boston_to_california(orig_desc)

            # If model DID NOT rewrite, we force a safe rewrite in the saved metadata
            if final_title == orig_title:
                final_title = _rewrite_boston_to_california(final_title)
            if final_desc == orig_desc:
                final_desc = _rewrite_boston_to_california(final_desc)

        else:
            # Not allowed: keep exactly original
            final_title = orig_title
            final_desc  = orig_desc

            # Warn if model tried to change them
            if parsed_title and parsed_title != orig_title:
                print(f"[warning] Item {idx}: model changed title.")
                print(f"  expected: {orig_title}")
                print(f"  got:      {parsed_title}")
            if parsed_desc and parsed_desc != orig_desc:
                print(f"[warning] Item {idx}: model changed description.")

        # Validate correct_code
        if not isinstance(correct_code, str) or not correct_code.strip():
            print(f"[warning] Item {idx}: correct_code is empty or not a string.")
            continue

        # Build entry (NOTE: base dataset unchanged; output file gets final_title/final_desc)
        codes_entry = {
            "title": final_title,
            "description": final_desc,
            "difficulty": orig_diff,   # always keep original difficulty
            "correct_code": correct_code,
        }
        codes_list.append(codes_entry)

        # Write updated codes file immediately
        if codes_output_path:
            try:
                with open(codes_output_path, "w", encoding="utf-8") as f:
                    json.dump(codes_list, f, ensure_ascii=False, indent=2)
                print(f"[codes] Appended item {idx} to {codes_output_path} (total {len(codes_list)})")
            except Exception as e:
                print(f"[warning] Failed to write {codes_output_path}: {e}")

        if sleep_between > 0:
            time.sleep(sleep_between)

    print("\n[done] Code generation finished.")
    if codes_output_path:
        try:
            with open(codes_output_path, "r", encoding="utf-8") as f:
                return json.load(f)
        except Exception:
            return codes_list
    return codes_list


In [ ]:
path = "test_dataset_codes_10_15.json"
with open(path, "w", encoding="utf-8") as f:
    json.dump([], f, ensure_ascii=False, indent=2)


In [ ]:
projects = generate_correct_code(
    model_id=MODELS["claude"],
    dataset= dataset,
    start_idx=550,
    end_idx=600,
    codes_output_path="dataset_codes_550_end.json"
)


Generating correct_code for items [550, 600) (total 50) using model: anthropic/claude-sonnet-4.5
Results will be appended to: dataset_codes_550_end.json

=== Project 550 / 599 ===
Title: California Housing Bayesian Optimization Stacking
Difficulty: hard
Latency: 35.14s | Raw length: 9267 chars
[codes] Appended item 550 to dataset_codes_550_end.json (total 1)

=== Project 551 / 599 ===
Title: California Housing LSTM Forecasting
Difficulty: hard
Latency: 15.73s | Raw length: 3533 chars
[codes] Appended item 551 to dataset_codes_550_end.json (total 2)

=== Project 552 / 599 ===
Title: California Housing Multi-Task Neural Regression
Difficulty: hard
Latency: 40.34s | Raw length: 10849 chars
[codes] Appended item 552 to dataset_codes_550_end.json (total 3)

=== Project 553 / 599 ===
Title: California Housing Stacked Ensemble
Difficulty: hard
Latency: 25.72s | Raw length: 5912 chars
[codes] Appended item 553 to dataset_codes_550_end.json (total 4)

=== Project 554 / 599 ===
Title: California

## 7. The Project's incorrect codes

### 7.1. System Prompt

In [9]:
INCORRECT_SYSTEM_PROMPT = """
Role: Python Code Perturbation Generator

You generate training data for code repair. You will receive:
1) a correct Python script (<correct_code>)
2) a target error label (<target_error_type>)

Goal
- Produce ONE incorrect variant of the given script that reliably triggers the requested error type.
- Keep the edit minimal (preferably 1–2 lines) and realistic.

Task
- Copy the provided correct code and apply EXACTLY ONE minimal edit.
- The modified script MUST trigger the requested <target_error_type> as the primary failure
  (or primary incorrectness for LogicError).
- Return ONLY a single RAW JSON object (not an array), with exactly two keys:
  - "incorrect_code": the FULL modified script (not a diff/patch; no placeholders)
  - "error_type": a single string formatted exactly as:
      "<Label>: <one-sentence reason> || line: <exact offending line>"
    The cited line MUST appear verbatim in incorrect_code.
    If the defect is a missing import line, use:
      "... || line: missing: <import ...>"

Allowed labels
SyntaxError, NameError, TypeError, ValueError, AttributeError, IndexError, KeyError, ImportError, LogicError

Hard rules
- Exactly ONE defect total. Do NOT add a second issue.
- Prefer modifying an EXISTING line over adding new lines.
- Do NOT append unrelated “crash lines” (e.g., print(undefined_variable), float("not_a_number"))
  unless there is no reasonable in-context edit to achieve the target label.
- Do NOT add new comments or explanatory text inside incorrect_code.
- Do NOT output placeholders like {CORRECT_CODE} or {ERROR_TYPE}.
- Do NOT return multiple candidates or an array.
- Do NOT wrap output in markdown/code fences/tags.
- If label is not SyntaxError, the code must remain syntactically valid Python.

Line-citation rules (very important)
- The "line:" must be the exact, complete line in incorrect_code where the error is raised
  (the call/usage site), not merely where a bad value was assigned.
- Copy the cited line verbatim from incorrect_code.
- Before output, verify that the cited "line:" is exactly the line that triggers the requested error when executed.

High-certainty, context-preserving edit patterns (use these first)
General preference:
- Modify an existing line in the script’s main execution path.

1) NameError (use-site, in-context)
- Introduce a 1–2 character typo at the usage site of an existing name (do not change its definition).
  Examples:
    grid_search.fit(...) -> gridsearch.fit(...)
    model.encode(xb) -> encoder(xb)

2) AttributeError (in-context)
- Introduce a 1–2 character typo in an existing attribute/method call on an existing object.
  Examples:
    plt.boxplot(...) -> plt.boxplotz(...)
    model.predict(X) -> model.predic(X)

3) ImportError (import line only)
- Misspell an imported module/symbol in an existing import statement (and leave the rest unchanged).
  Example:
    from sklearn.neighbors import ... -> from sklearn.neigbors import ...

4) TypeError (in-context, reliable)
- Remove a required argument from an existing function call.
  Example:
    train_test_split(X, y, ...) -> train_test_split(X)
- Or pass an obviously wrong type to a strictly validated parameter already present.
  Example (often TypeError):
    plt.scatter(..., s=10) -> plt.scatter(..., s="10")

5) ValueError (in-context, highly reliable)
Choose the most natural one that already exists in the script:
- train_test_split: set test_size outside (0, 1)
  Example:
    test_size=0.2 -> test_size=1.5
- cross_val_score / CV: set cv to 0 or 1 (must be >= 2)
  Example:
    cv=5 -> cv=0
Avoid unrelated ValueErrors like float("not_a_number") unless no in-context option exists.

6) IndexError (in-context)
- Change an existing index access to be out of range.
  Examples:
    arr[0] -> arr[999999]
    lst[-1] -> lst[len(lst)]

7) KeyError (in-context)
- If dict access exists, slightly misspell an existing key string.
  Example:
    d["accuracy"] -> d["acuracy"]
- If no dict access exists, do NOT invent new dict logic; only add one minimal dict access if the label is forced.

8) SyntaxError (single-line, reliable)
- Remove a closing parenthesis on an existing call line, OR remove a required colon on an existing block header.
  Examples:
    print("x") -> print("x"
    if cond: -> if cond
- Do not change multiple lines.

Self-check before final output
- Exactly ONE defect is present.
- The error matches <target_error_type>.
- The cited "line:" is the true failing line (call/usage site) and appears verbatim in incorrect_code.
- Output is a single valid JSON object with exactly two keys: incorrect_code, error_type.
""".strip()


### 7.2. Few Shot Examples

In [10]:
INCORRECT_FEW_SHOTS= r"""
Example 1 — Iris KNN Classifier (sklearn, stratified split) | target=ImportError | 1 defect:
  Input:
    <correct_code>
import argparse
import sys
import random
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

def main():
    p = argparse.ArgumentParser(description="Iris KNN classifier with k=3; prints TEST_PASS if accuracy >= 0.9.")
    p.add_argument("--test-size", type=float, default=0.2, help="Test set fraction (default: 0.2).")
    p.add_argument("--k", type=int, default=3, help="Number of neighbors for KNN (default: 3).")
    p.add_argument("--seed", type=int, default=42, help="Random seed (default: 42).")
    args = p.parse_args()

    random.seed(args.seed)
    np.random.seed(args.seed)

    data = load_iris()
    X, y = data.data, data.target

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=args.test_size, random_state=args.seed, stratify=y
    )

    knn = KNeighborsClassifier(n_neighbors=args.k)
    knn.fit(X_train, y_train)

    accuracy = knn.score(X_test, y_test)
    print(f"Test accuracy: {accuracy:.3f}")

    if accuracy >= 0.9:
        print("TEST_PASS")
    else:
        print(f"TEST_FAIL: accuracy {accuracy:.3f} < 0.9")
        sys.exit(1)

if __name__ == "__main__":
    main()
    </correct_code>
    <target_error_type>ImportError</target_error_type>

  Output:
    {"incorrect_code":"import argparse\nimport sys\nimport random\nimport numpy as np\nfrom sklearn.datasets import load_iris\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.neigbors import KNeighborsClassifier\n\ndef main():\n    p = argparse.ArgumentParser(description=\"Iris KNN classifier with k=3; prints TEST_PASS if accuracy >= 0.9.\")\n    p.add_argument(\"--test-size\", type=float, default=0.2, help=\"Test set fraction (default: 0.2).\")\n    p.add_argument(\"--k\", type=int, default=3, help=\"Number of neighbors for KNN (default: 3).\")\n    p.add_argument(\"--seed\", type=int, default=42, help=\"Random seed (default: 42).\")\n    args = p.parse_args()\n\n    random.seed(args.seed)\n    np.random.seed(args.seed)\n\n    data = load_iris()\n    X, y = data.data, data.target\n\n    X_train, X_test, y_train, y_test = train_test_split(\n        X, y, test_size=args.test_size, random_state=args.seed, stratify=y\n    )\n\n    knn = KNeighborsClassifier(n_neighbors=args.k)\n    knn.fit(X_train, y_train)\n\n    accuracy = knn.score(X_test, y_test)\n    print(f\"Test accuracy: {accuracy:.3f}\")\n\n    if accuracy >= 0.9:\n        print(\"TEST_PASS\")\n    else:\n        print(f\"TEST_FAIL: accuracy {accuracy:.3f} < 0.9\")\n        sys.exit(1)\n\nif __name__ == \"__main__\":\n    main()","error_type":"ImportError: The import fails because the sklearn module path is misspelled. || line: from sklearn.neigbors import KNeighborsClassifier"}


Example 2 — Torch VAE Latent Scatter (FakeData) | target=NameError | 1 defect:
  Input:
    <correct_code>
import argparse
import os
import random
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def main():
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import DataLoader
    from torchvision.datasets import FakeData
    from torchvision import transforms

    p = argparse.ArgumentParser(description="Train a tiny VAE on FakeData and plot 2D latent means.")
    p.add_argument("--epochs", type=int, default=2)
    p.add_argument("--batch", type=int, default=128)
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--output", type=str, default="vae_latent.png")
    args = p.parse_args()

    random.seed(args.seed)
    np.random.seed(args.seed)
    torch.manual_seed(args.seed)

    tfm = transforms.ToTensor()
    ds = FakeData(size=1000, image_size=(1, 28, 28), num_classes=10, transform=tfm)
    loader = DataLoader(ds, batch_size=args.batch, shuffle=True)

    class VAE(nn.Module):
        def __init__(self, latent_dim=2):
            super().__init__()
            self.enc = nn.Sequential(
                nn.Flatten(),
                nn.Linear(28*28, 128),
                nn.ReLU(),
            )
            self.mu = nn.Linear(128, latent_dim)
            self.logvar = nn.Linear(128, latent_dim)
            self.dec = nn.Sequential(
                nn.Linear(latent_dim, 128),
                nn.ReLU(),
                nn.Linear(128, 28*28),
                nn.Sigmoid(),
            )
            self.latent_dim = latent_dim

        def encode(self, x):
            h = self.enc(x)
            return self.mu(h), self.logvar(h)

        def reparameterize(self, mu, logvar):
            import torch
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            return mu + eps * std

        def decode(self, z):
            return self.dec(z).view(-1, 1, 28, 28)

        def forward(self, x):
            mu, logvar = self.encode(x)
            z = self.reparameterize(mu, logvar)
            recon = self.decode(z)
            return recon, mu, logvar

    def loss_fn(recon, x, mu, logvar):
        import torch
        import torch.nn.functional as F
        bce = F.binary_cross_entropy(recon, x, reduction="sum")
        kld = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
        return bce + kld

    model = VAE(latent_dim=2)
    opt = optim.Adam(model.parameters(), lr=1e-3)

    model.train()
    for _ in range(args.epochs):
        for xb, _ in loader:
            opt.zero_grad()
            recon, mu, logvar = model(xb)
            loss = loss_fn(recon, xb, mu, logvar)
            loss.backward()
            opt.step()

    model.eval()
    latents = []
    labels = []
    with torch.no_grad():
        for xb, yb in loader:
            mu, _ = model.encode(xb)
            latents.append(mu.cpu().numpy())
            labels.append(yb.cpu().numpy())
    latents = np.concatenate(latents, axis=0)
    labels = np.concatenate(labels, axis=0)

    plt.figure(figsize=(8, 6))
    plt.scatter(latents[:, 0], latents[:, 1], c=labels, alpha=0.6, s=10)
    plt.tight_layout()
    plt.savefig(args.output, dpi=120)
    plt.close()

    if os.path.isfile(args.output):
        print("TEST_PASS")
    else:
        raise RuntimeError("Output file not created")

if __name__ == "__main__":
    main()
    </correct_code>
    <target_error_type>NameError</target_error_type>

  Output:
    {"incorrect_code":"import argparse\nimport os\nimport random\nimport numpy as np\nimport matplotlib\nmatplotlib.use(\"Agg\")\nimport matplotlib.pyplot as plt\n\ndef main():\n    import torch\n    import torch.nn as nn\n    import torch.optim as optim\n    from torch.utils.data import DataLoader\n    from torchvision.datasets import FakeData\n    from torchvision import transforms\n\n    p = argparse.ArgumentParser(description=\"Train a tiny VAE on FakeData and plot 2D latent means.\")\n    p.add_argument(\"--epochs\", type=int, default=2)\n    p.add_argument(\"--batch\", type=int, default=128)\n    p.add_argument(\"--seed\", type=int, default=42)\n    p.add_argument(\"--output\", type=str, default=\"vae_latent.png\")\n    args = p.parse_args()\n\n    random.seed(args.seed)\n    np.random.seed(args.seed)\n    torch.manual_seed(args.seed)\n\n    tfm = transforms.ToTensor()\n    ds = FakeData(size=1000, image_size=(1, 28, 28), num_classes=10, transform=tfm)\n    loader = DataLoader(ds, batch_size=args.batch, shuffle=True)\n\n    class VAE(nn.Module):\n        def __init__(self, latent_dim=2):\n            super().__init__()\n            self.enc = nn.Sequential(\n                nn.Flatten(),\n                nn.Linear(28*28, 128),\n                nn.ReLU(),\n            )\n            self.mu = nn.Linear(128, latent_dim)\n            self.logvar = nn.Linear(128, latent_dim)\n            self.dec = nn.Sequential(\n                nn.Linear(latent_dim, 128),\n                nn.ReLU(),\n                nn.Linear(128, 28*28),\n                nn.Sigmoid(),\n            )\n            self.latent_dim = latent_dim\n\n        def encode(self, x):\n            h = self.enc(x)\n            return self.mu(h), self.logvar(h)\n\n        def reparameterize(self, mu, logvar):\n            import torch\n            std = torch.exp(0.5 * logvar)\n            eps = torch.randn_like(std)\n            return mu + eps * std\n\n        def decode(self, z):\n            return self.dec(z).view(-1, 1, 28, 28)\n\n        def forward(self, x):\n            mu, logvar = self.encode(x)\n            z = self.reparameterize(mu, logvar)\n            recon = self.decode(z)\n            return recon, mu, logvar\n\n    def loss_fn(recon, x, mu, logvar):\n        import torch\n        import torch.nn.functional as F\n        bce = F.binary_cross_entropy(recon, x, reduction=\"sum\")\n        kld = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())\n        return bce + kld\n\n    model = VAE(latent_dim=2)\n    opt = optim.Adam(model.parameters(), lr=1e-3)\n\n    model.train()\n    for _ in range(args.epochs):\n        for xb, _ in loader:\n            opt.zero_grad()\n            recon, mu, logvar = model(xb)\n            loss = loss_fn(recon, xb, mu, logvar)\n            loss.backward()\n            opt.step()\n\n    model.eval()\n    latents = []\n    labels = []\n    with torch.no_grad():\n        for xb, yb in loader:\n            mu, _ = encoder(xb)\n            latents.append(mu.cpu().numpy())\n            labels.append(yb.cpu().numpy())\n    latents = np.concatenate(latents, axis=0)\n    labels = np.concatenate(labels, axis=0)\n\n    plt.figure(figsize=(8, 6))\n    plt.scatter(latents[:, 0], latents[:, 1], c=labels, alpha=0.6, s=10)\n    plt.tight_layout()\n    plt.savefig(args.output, dpi=120)\n    plt.close()\n\n    if os.path.isfile(args.output):\n        print(\"TEST_PASS\")\n    else:\n        raise RuntimeError(\"Output file not created\")\n\nif __name__ == \"__main__\":\n    main()","error_type":"NameError: This occurs because 'encoder' is called but never defined. || line: mu, _ = encoder(xb)"}
""".strip()


### 7.2. User Prompt

In [11]:
INCORRECT_USER_PROMPT = """
Modify the following correct Python code by applying exactly ONE minimal edit so that executing it triggers the requested target error type.

Return ONLY a single JSON object with keys: "incorrect_code" and "error_type".
- incorrect_code must be the FULL modified script.
- error_type must be: "<Label>: <one-sentence reason> || line: <exact offending line>"

<correct_code>
{{CORRECT_CODE}}
</correct_code>

<target_error_type>
{{ERROR_TYPE}}
</target_error_type>
""".strip()

### 7.3. Generating the Incorrect codes

In [12]:
def extract_json_incorrect(text: str):
    """
    Extract a JSON object from model output.
    NEVER raises (returns None on failure) so your run won't stop.
    """
    if not isinstance(text, str) or not text.strip():
        return None

    # 1) If fenced, extract inner content
    m = re.search(r"```(?:json)?\s*([\s\S]*?)\s*```", text, flags=re.IGNORECASE)
    if m:
        text = m.group(1).strip()

    # 2) Try raw_decode from any '{'
    dec = json.JSONDecoder()
    i = text.find("{")
    while i != -1:
        try:
            obj, _end = dec.raw_decode(text, i)
            # Normalize list->dict if needed
            if isinstance(obj, list) and obj and isinstance(obj[0], dict):
                obj = obj[0]
            if isinstance(obj, dict):
                return obj
        except Exception:
            pass
        i = text.find("{", i + 1)

    # 3) Try parsing full text as JSON (object or list)
    try:
        obj = json.loads(text)
        if isinstance(obj, dict):
            return obj
        if isinstance(obj, list) and obj and isinstance(obj[0], dict):
            return obj[0]
    except Exception:
        pass

    # 4) Give up safely
    return None



In [13]:
def _normalize_parsed_json(parsed):
    """Accept dict or [dict] and normalize to a dict."""
    if isinstance(parsed, dict):
        return parsed
    if isinstance(parsed, list) and len(parsed) > 0 and isinstance(parsed[0], dict):
        return parsed[0]
    return None


def build_single_error_messages(correct_code: str, target_error_type: str):
    filled_user = (
        INCORRECT_USER_PROMPT
        .replace("{{CORRECT_CODE}}", correct_code)
        .replace("{{ERROR_TYPE}}", target_error_type)
    )
    # Few-shots as an assistant message is usually fine in chat APIs
    return [
        {"role": "system", "content": INCORRECT_SYSTEM_PROMPT},
        {"role": "assistant", "content": INCORRECT_FEW_SHOTS},
        {"role": "user", "content": filled_user},
    ]



In [14]:
def generate_incorrect_codes_single_error(
    *,
    model_id: str,
    input_path: str,
    output_path: str,
    error_types: list,
    temperature: float = 0.2,
    top_p: float = 0.9,
    max_tokens: int = 3000,
    sleep_between: float = 0.5,
    limit: int | None = None,
    start_idx: int = 0,
    choose_random_error: bool = True,
    checkpoint_every: int = 10,
    seed: int | None = None,
    max_retries: int = 3,
    retry_sleep: float = 1.0,
    save_raw_on_fail: bool = True
):
    if seed is not None:
        random.seed(seed)

    with open(input_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        raise ValueError("Input JSON must be a list of samples (JSON array).")

    end_idx = len(data) if limit is None else min(len(data), start_idx + limit)

    for idx in range(start_idx, end_idx):
        sample = data[idx]
        correct_code = sample.get("correct_code", "")
        if not isinstance(correct_code, str) or not correct_code.strip():
            sample["incorrect_code"] = ""
            sample["error_type"] = "MISSING_CORRECT_CODE: sample has no correct_code. || line: missing: correct_code"
            continue

        if choose_random_error:
            target_error = random.choice(error_types)
        else:
            target_error = sample.get("requested_error_type")
            if target_error not in error_types:
                target_error = random.choice(error_types)

        messages = build_single_error_messages(correct_code, target_error)

        parsed = None
        raw_text = ""
        latency = 0.0

        # NEW: retry loop (prevents hard stop + reduces failures)
        for attempt in range(1, max_retries + 1):
            raw_text, latency = call_openrouter_model(
                model_id=model_id,
                messages=messages,
                temperature=temperature,
                top_p=top_p,
                max_tokens=max_tokens,
            )
            print(f"[{idx}] target={target_error} | attempt={attempt}/{max_retries} | latency={latency:.2f}s | raw_len={len(raw_text)}")

            parsed = extract_json_incorrect(raw_text)
            if isinstance(parsed, dict):
                parsed = _normalize_parsed_json(parsed)
                # Accept only if required keys exist (prevents junk JSON)
                if isinstance(parsed, dict) and (parsed.get("incorrect_code") is not None) and (parsed.get("error_type") is not None):
                    break
            parsed = None
            if retry_sleep:
                time.sleep(retry_sleep)

        if not isinstance(parsed, dict):
            # NEW: do not crash; store failure info and continue
            sample["incorrect_code"] = ""
            sample["error_type"] = f"{target_error}: PARSE_FAILED || line: missing: json_object"
            if save_raw_on_fail:
                sample["raw_model_output"] = raw_text
        else:
            sample["incorrect_code"] = (parsed.get("incorrect_code") or "").strip()
            sample["error_type"] = (parsed.get("error_type") or "").strip()

        if checkpoint_every and ((idx + 1) % checkpoint_every == 0):
            with open(output_path, "w", encoding="utf-8") as f:
                json.dump(data, f, ensure_ascii=False, indent=2)
            print(f"[checkpoint] saved -> {output_path}")

        if sleep_between:
            time.sleep(sleep_between)

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    return data


In [19]:
ERROR_TYPES = [
    "SyntaxError", "NameError", "TypeError", "ValueError",
    "AttributeError", "IndexError", "KeyError", "ImportError",
    "LogicError",
]

output = generate_incorrect_codes_single_error(
    model_id=MODELS["gpt"],
    input_path="dataset_codes_50_100_with_incorrect.json",
    output_path="dataset_codes_50_100_with_incorrect.json",
    error_types=ERROR_TYPES,
    limit=15,
    start_idx= 0,
    choose_random_error=True,
    checkpoint_every= 5,
    seed=42,
)


[0] target=NameError | attempt=1/3 | latency=26.16s | raw_len=2868
[1] target=SyntaxError | attempt=1/3 | latency=12.72s | raw_len=2170
[2] target=AttributeError | attempt=1/3 | latency=18.05s | raw_len=2724
[3] target=ValueError | attempt=1/3 | latency=15.92s | raw_len=2338
[4] target=ValueError | attempt=1/3 | latency=4.76s | raw_len=2615
[checkpoint] saved -> dataset_codes_50_100_with_incorrect.json
[5] target=TypeError | attempt=1/3 | latency=25.48s | raw_len=3862
[6] target=NameError | attempt=1/3 | latency=20.53s | raw_len=2692
[7] target=LogicError | attempt=1/3 | latency=9.07s | raw_len=1720
[8] target=NameError | attempt=1/3 | latency=11.58s | raw_len=4652
[9] target=KeyError | attempt=1/3 | latency=6.77s | raw_len=2728
[checkpoint] saved -> dataset_codes_50_100_with_incorrect.json
[10] target=SyntaxError | attempt=1/3 | latency=27.89s | raw_len=1899
[11] target=SyntaxError | attempt=1/3 | latency=7.83s | raw_len=2209
[12] target=NameError | attempt=1/3 | latency=12.40s | raw_

In [3]:
with open("final_dataset.json", "r", encoding= "utf-8") as f:
  final_dataset = json.load(f)

In [6]:
print(f"Number of samples in Final Dataset is: {len(final_dataset)} samples")

Number of samples in Final Dataset is: 582 samples


## 8. Conclusion: Dataset Ready for Finetuning

So far, we generated 600 project samples. After a quality review (syntax, executability, and alignment with the descriptions), we removed the problematic ones and ended up with 582 samples that are suitable for finetuning.

**What the final dataset contains**

Each sample includes:

* title: the project title

* description: the project requirements and expected output

* difficulty: one of [easy, medium, hard]

* correct_code: a complete, runnable Python script that satisfies the description

* incorrect_code: the same script with exactly one minimal defect introduced

* error_type: the error label plus a short reason and the exact line where the error occurs

**Why we generated this dataset**

This dataset is designed to teach a smaller language model two closely related skills:

1. Code generation from a natural-language project description (via the `correct_code`).

2. Code repair / autocorrection by learning to recognize common failure modes and fix them (via paired `incorrect_code` + `error_type`).

By exposing the model to many realistic single-error variants, it learns patterns for diagnosing and correcting